# EXP-01 · Quantus FC: same explanation, different estimates

**질문:** 같은 설명을 그대로 평가할 때, subset 크기 $k$를 바꾸면 population FC가 바뀔까, 아니면 일부 $B$개 draw로 계산한 추정값만 달라질까?

**Quantus 0.6.0 public FC/FE calls produce the metric scores.** This notebook installs an isolated CPU environment and runs the same package through the CLI. NumPy references and trace replay check the mathematical target and the implementation; they never replace the reported Quantus score.

Use a **fresh CPU runtime**. The fixed 20-feature model needs no training, labels, external dataset or GPU. Start with smoke; full development and reserved confirmation are separate stages. [Design and evidence](https://github.com/theguardianofthebed/xai-metric-behavior/blob/exp01-atlas-protocol/docs/EXP-01.md).


## 1. Install the isolated experiment environment

**Colab:** start in a fresh CPU runtime. This cell clones `exp01-atlas-protocol` if needed, then creates or reuses the repository's `.venv`. **Local Jupyter:** start inside the repository or set `REPO_DIR` below. An existing checkout is never updated silently; its commit is printed.

The first install can take several minutes. The isolated environment contains required CPU PyTorch, Quantus and the package dependencies. Colab's preinstalled optional ML frameworks and already-loaded NumPy stay outside it; the running notebook kernel is not upgraded.


In [ ]:
from pathlib import Path
import os, sys, subprocess, json, tomllib

REPOSITORY = "https://github.com/theguardianofthebed/xai-metric-behavior.git"
REF = "exp01-atlas-protocol"
REPO_DIR = None  # Optionally set the path of an existing checkout.

if REPO_DIR:
    repo = Path(REPO_DIR).expanduser().resolve()
else:
    repo = next((p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / "configs/exp01_smoke.yaml").exists()
                 and (p / "pyproject.toml").exists()), None)
    if repo is None:
        repo = (Path("/content") if Path("/content").exists() else Path.cwd()) / "xai-metric-behavior"
        if not repo.exists():
            subprocess.run(["git", "clone", "--branch", REF, REPOSITORY, str(repo)], check=True)
if not (repo / "configs/exp01_smoke.yaml").exists():
    raise RuntimeError("Use a checkout containing the EXP-01 Quantus implementation.")
project = tomllib.loads((repo / "pyproject.toml").read_text())["project"]
if not any(req.lower().startswith("quantus") for req in project["dependencies"]):
    raise RuntimeError("This checkout predates the Quantus migration. Update it explicitly or use a fresh clone.")
os.chdir(repo)
env_dir = repo / ".venv"
env_python = env_dir / ("Scripts/python.exe" if os.name == "nt" else "bin/python")
if not env_python.exists():
    subprocess.run([sys.executable, "-m", "venv", str(env_dir)], check=True)
subprocess.run([str(env_python), "-m", "pip", "install", "--quiet", "torch==2.10.0",
                "--index-url", "https://download.pytorch.org/whl/cpu"], check=True)
subprocess.run([str(env_python), "-m", "pip", "install", "--quiet", "-e", str(repo)], check=True)
version_query = (
    "import sys,json,importlib.metadata as m; "
    "print(json.dumps({'Python':sys.version.split()[0], "
    "**{p:m.version(p) for p in ['xai-metric-behavior','quantus','torch','numpy','scipy']}}))"
)
versions = json.loads(subprocess.check_output([str(env_python), "-c", version_query], text=True))
print("Repository:", repo)
print("Commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())
print("Experiment interpreter:", env_python)
print(json.dumps(versions, indent=2))


## 2. Read the contract and choose the run

- The scalar target is $g(x)=2.5x_1-2x_2+1.5x_3+x_4-0.5x_5$ on 20 features. Only five affect the output.
- Literal baseline **0** defines the signed exact contributions. No attribution normalization, absolute-value conversion or softmax is applied.
- **Exact** is a calibration anchor. **A** moves contribution 1 to unused feature 6; **B** also moves contribution 2 to unused feature 7. Their value distribution stays the same while locations change.
- **$k$** counts features per draw; **$B$** counts draws per score; **$R$** counts independent evaluator repeats per input; **$N$** counts independent inputs.
- Quantus samples natively. The adapter records actual draws and checks pairing across candidates, k and B. It preserves float64 inputs with a scoped setting and restores caller RNG/dtype state.

Under the declared additive, uniform-subset assumptions, $FC_{\mathrm{pop}}(k)=\mathrm{Corr}_j(a_j,v_j)$ for positive variances and $1\leq k<20$. Finite scores can still vary or be undefined.


In [ ]:
PROFILE = "smoke"  # Change to "development" only after reviewing smoke.
if PROFILE not in {"smoke", "development"}:
    raise ValueError("Reserved confirmation uses the separate freeze step below.")
config_path = repo / "configs" / f"exp01_{PROFILE}.yaml"
print(config_path.read_text())  # The experiment interpreter parses and validates this YAML.


## 3. Run the official Quantus path

A timestamp creates a new output directory. The isolated interpreter runs the package CLI, which calls the official metrics, records their calculations, checks references/pairing, and saves summaries and figures. The full development grid can take substantially longer than smoke.

A returned nonfinite correlation is preserved with its reason. The notebook does not fill it with zero or substitute another metric implementation.


In [ ]:
from datetime import datetime, timezone
from time import perf_counter

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
output_dir = repo / "results" / f"exp01-{PROFILE}-{stamp}"
started = perf_counter()
subprocess.run([str(env_python), "-m", "xai_metric_behavior", "run",
                "--config", str(config_path), "--output", str(output_dir)],
               cwd=repo, check=True)
print("Saved:", output_dir)
print(f"Elapsed: {perf_counter() - started:.1f} seconds")


## 4. Inspect scores and failures together

`manifest.json` identifies this run and its checks. `population_scores.csv` supplies the analytical reference and official singleton FE check. `scores.csv` contains returned Quantus FC values; `failures.csv` retains undefined/error outcomes. Paired contrasts require common validity, so changing failure rates can change which cases contribute.

**Read the plots:** population FC versus k shows the **first input only**; bias/RMSE versus B summarizes inputs separately by k; undefined fraction versus B uses **all attempts** in that condition. A smaller error concerns estimator behavior, not universal explanation quality.


In [ ]:
import csv, json
from IPython.display import display, Markdown, Image

manifest = json.loads((output_dir / "manifest.json").read_text())
print(json.dumps({key: manifest[key] for key in
                 ["experiment_id", "phase", "status", "run_id", "metric_variant", "library_execution", "backend_execution_sha256", "counts", "validation"]
                 if key in manifest}, indent=2))

def preview_csv(name, rows=6, columns=None):
    with (output_dir / name).open(newline="") as stream:
        reader = csv.DictReader(stream)
        fields = reader.fieldnames or []
        records = list(reader)
    chosen = [f for f in (columns or fields[:6]) if f in fields]
    display(Markdown(f"**{name}** — {len(records)} rows; showing up to {rows}."))
    if not records or not chosen:
        return
    def escape(value):
        return str(value).replace("|", r"\|").replace("\n", " ")
    table = "| " + " | ".join(chosen) + " |\n"
    table += "| " + " | ".join(["---"] * len(chosen)) + " |\n"
    table += "\n".join("| " + " | ".join(escape(row.get(f, "")) for f in chosen) + " |"
                       for row in records[:rows])
    display(Markdown(table))

preview_csv("population_scores.csv", columns=["input_id", "k", "explanation", "population_FC", "singleton_FE", "valid"])
preview_csv("scores.csv", columns=["input_id", "repetition", "k", "budget", "explanation", "score", "valid", "reason"])
preview_csv("contrast_summary.csv")
preview_csv("failures.csv", columns=["stage", "input_id", "repetition", "explanation", "k", "budget", "category", "reason"])
for figure in sorted((output_dir / "figures").glob("*.png")):
    display(Markdown(f"### {figure.stem.replace('_', ' ')}"))
    display(Image(filename=str(figure)))


## 5. Follow the recorded computation

For each official draw, $U_S$ is the supplied attribution sum on the selected coordinates, and $V_S=g(x)-g(x_{S\leftarrow0})$ is the output change from replacing those coordinates in the **original input**. Quantus computes Pearson within one input, across draws.

The engine already checks trace parity. FC arrays are in `input_*.npz`; separate `FE_input_*.npz` files retain official singleton traces. `metric_calls.jsonl` records every public attempt, returned score and check. The cell below only inspects saved arrays; it does not calculate a replacement score. Prefix pairing means that related budgets reuse draws, not that they are independent experiments.


In [ ]:
import numpy as np

trace_path = sorted((output_dir / "traces").glob("input_*.npz"))[0]
with np.load(trace_path, allow_pickle=False) as trace:
    print("Trace:", trace_path.name)
    print("Saved array shapes:", {key: trace[key].shape for key in trace.files})
    keys = [key for key in trace.files
            if key.startswith(("subsets", "indices", "masks", "U", "V", "perturbed", "official"))]
    for key in keys[:6]:
        print(key, "— first recorded values:", trace[key].reshape(-1)[:12])


## 6. Review development, then freeze confirmation

After inspecting smoke, rerun with `PROFILE="development"`. Review estimator errors, population margins, failure rates and paired contrasts before freezing the prediction. Counts are planning choices, not a power calculation.

Use the completed development directory printed by the notebook. From the repository, run these commands with the isolated interpreter (prefix each with `!` in a Colab code cell):

```bash
.venv/bin/python -m xai_metric_behavior freeze --config configs/exp01_confirmation.yaml --development-run results/YOUR_DEVELOPMENT_RUN --output protocol-freeze.json
.venv/bin/python -m xai_metric_behavior run --config configs/exp01_confirmation.yaml --freeze protocol-freeze.json --output results/exp01-confirmation
```

Reserved subset sizes are `{2,7,13}`, with fresh inputs and draw streams. The freeze binds configuration, implementation, development provenance, and the backend execution hash (package versions plus eight selected Quantus source files). Revising a prediction after seeing confirmation requires fresh reserved conditions.

Download or archive the **complete output directory**, together with its freeze file when applicable. EXP-01-R remains an Atlas plan only; this notebook does not download real data or train its proposed model.


## Research Learning Summary

**필요성 →** 같은 설명의 점수 변화가 sampling 때문인지 구분해야 한다.  
**직관적인 예시/이해 →** 채점에 사용하는 feature 묶음과 확인 횟수만 바꾼다. 모든 허용 subset의 기준값은 같아도 일부 표본의 점수는 달라질 수 있다.  
**공식적인 정의 →** population FC는 전체 subset 분포의 Pearson, finite FC는 B개 draw의 Pearson이다. Quantus public call의 반환값이 실제 보고 점수이며, reference와 trace replay는 보조 점검이다.  
**근거/그게 되는 이유 →** additive model과 uniform fixed-size sampling에서는 공분산과 두 분산의 공통 계수가 population Pearson에서 상쇄된다. Finite sample의 변동·퇴화까지 없어지지는 않는다. Parity 통과만으로 일반적인 metric 신뢰성이 증명되지는 않는다.
